In [2]:
import pandas as pd
import json
import ast
import random
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Cargar el archivo JSONL original
data = []
with open("ragtruth_qa_filtrado_es.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

df_original = pd.DataFrame(data)
print(f"Dataset original cargado con éxito. Total de filas: {len(df_original)}")

Dataset original cargado con éxito. Total de filas: 5913


In [3]:
# Función para limpiar y parsear las etiquetas de alucinación
def parsear_etiquetas(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return x if isinstance(x, list) else []

df_original['parsed_labels'] = df_original['labels_originales'].apply(parsear_etiquetas)
df_original['tiene_alucinacion'] = df_original['parsed_labels'].apply(lambda x: len(x) > 0)

# Mostrar estadísticas del EDA inicial
print("=== EDA 1: ESTADO DEL DATASET ORIGINAL ===")
print(df_original['tiene_alucinacion'].value_counts(normalize=True).map(lambda x: f"{x:.2%}"))
print(f"\nTotal Sin Alucinación: {sum(~df_original['tiene_alucinacion'])}")
print(f"Total Con Alucinación: {sum(df_original['tiene_alucinacion'])}")

=== EDA 1: ESTADO DEL DATASET ORIGINAL ===
tiene_alucinacion
False    70.95%
True     29.05%
Name: proportion, dtype: str

Total Sin Alucinación: 4195
Total Con Alucinación: 1718


In [4]:
def transformar_fila(row):
    etiquetas = row['parsed_labels']
    tiene_aluc = row['tiene_alucinacion']
    
    # Moneda al aire para elegir el idioma (50/50)
    idioma = "es" if random.random() < 0.5 else "en"
    
    if idioma == "es":
        prompt = row["prompt_es"]
        respuesta = row["response_es"]
        
        # SACANDO TODO EL PROVECHO A LOS DATOS EN ESPAÑOL
        if tiene_aluc:
            tipo_aluc = etiquetas[0].get("label_type", "Alucinación general")
            frase_inventada = etiquetas[0].get("text", "Texto no especificado")
            
            veredicto = (f"Sí, se detectó una alucinación del tipo '{tipo_aluc}'. "
                         f"El modelo generó la siguiente información sin respaldo en el contexto: \"{frase_inventada}\".")
        else:
            veredicto = "No, la respuesta es correcta, segura y está totalmente respaldada por los pasajes del contexto."
            
        texto_final = f"### Tarea: Analiza si la siguiente respuesta contiene alucinaciones basándote en el contexto.\n\n### Contexto y Pregunta:\n{prompt}\n\n### Respuesta a evaluar:\n{respuesta}\n\n### Veredicto del Auditor:\n{veredicto}"
        
    else:
        # SACANDO TODO EL PROVECHO A LOS DATOS EN INGLÉS
        prompt = row["prompt_en"]
        respuesta = row["response_en"]
        
        if tiene_aluc:
            tipo_aluc = etiquetas[0].get("label_type", "General Hallucination")
            frase_inventada = etiquetas[0].get("text", "Unspecified text")
            
            veredicto = (f"Yes, a hallucination of type '{tipo_aluc}' was detected. "
                         f"The model generated the following unsupported information: \"{frase_inventada}\".")
        else:
            veredicto = "No, the response is correct, safe, and fully supported by the context passages."
            
        texto_final = f"### Task: Analyze if the following response contains hallucinations based on the context.\n\n### Context and Question:\n{prompt}\n\n### Response to evaluate:\n{respuesta}\n\n### Auditor Verdict:\n{veredicto}"
    
    return pd.Series({'text': texto_final, 'idioma_asignado': idioma, 'target_alucinacion': tiene_aluc})

# Aplicar la transformación para generar el nuevo dataset estructurado
df_procesado = df_original.apply(transformar_fila, axis=1)
print("✨ Transformación completada con extracción de frases alucinadas.")

✨ Transformación completada con extracción de frases alucinadas.


In [6]:
print("=== EDA 2: VALIDACIÓN DEL NUEVO DATASET ===")
print("\nDistribución de Idiomas asignados:")
print(df_procesado['idioma_asignado'].value_counts())

print("\nDistribución de Alucinaciones en el nuevo texto:")
print(df_procesado['target_alucinacion'].value_counts())

# Mostrar un ejemplo aleatorio para revisar visualmente la estructura
print("\nEJEMPLO DEL TEXTO QUE VERÁ EL LLM:\n")
print(df_procesado['text'].iloc[random.randint(0, len(df_procesado)-1)])

=== EDA 2: VALIDACIÓN DEL NUEVO DATASET ===

Distribución de Idiomas asignados:
idioma_asignado
es    2962
en    2951
Name: count, dtype: int64

Distribución de Alucinaciones en el nuevo texto:
target_alucinacion
False    4195
True     1718
Name: count, dtype: int64

EJEMPLO DEL TEXTO QUE VERÁ EL LLM:

### Tarea: Analiza si la siguiente respuesta contiene alucinaciones basándote en el contexto.

### Contexto y Pregunta:
Responde brevemente a la siguiente pregunta:
cómo funciona un televisor de pantalla plana
Ten en cuenta que tu respuesta debe basarse estrictamente en los siguientes tres pasajes:
pasaje 1: La configuración. Entre dos láminas de vidrio se encuentra la tecnología que permite que funcionen los televisores de pantalla plana. Dependiendo del tipo de televisor de pantalla plana, el proceso puede diferir ligeramente. En un televisor de plasma hay celdas que contienen los gases xenón y neón, además de electrodos. Hay dos tipos de electrodos, de visualización y de dirección, qu

In [ ]:
# === MUESTREO BALANCEADO: 400 muestras por idioma ===
# Separar por idioma del df ya transformado
df_es = df_procesado[df_procesado['idioma_asignado'] == 'es']
df_en = df_procesado[df_procesado['idioma_asignado'] == 'en']

print(f"Disponibles — ES: {len(df_es)} | EN: {len(df_en)}")

# Tomar exactamente 400 por idioma para entrenamiento
train_es = df_es.sample(n=400, random_state=42)
train_en = df_en.sample(n=400, random_state=42)
train_df = pd.concat([train_es, train_en]).sample(frac=1, random_state=42).reset_index(drop=True)

# El resto lo usamos para validación (mezclado, sin límite)
resto_es = df_es.drop(train_es.index)
resto_en = df_en.drop(train_en.index)
val_df = pd.concat([resto_es, resto_en]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nTamaños finales:")
print(f"  Train  : {len(train_df)} filas  (ES={len(train_es)}, EN={len(train_en)})")
print(f"  Val    : {len(val_df)} filas")

# Verificar distribución de idiomas en train
print(f"\nDistribución de idiomas en Train:")
print(train_df['idioma_asignado'].value_counts())

# Guardar los archivos JSONL para el script de entrenamiento
train_df[['text']].to_json("ragtruth_train.jsonl", orient="records", lines=True, force_ascii=False)
val_df[['text']].to_json("ragtruth_val.jsonl", orient="records", lines=True, force_ascii=False)

print("\n¡Archivos guardados: 'ragtruth_train.jsonl' (800) y 'ragtruth_val.jsonl'!")